# report3 — Sionna RT 광선 반사 실험

> ### ❓ 이 리포트가 답하는 질문
> **Sionna RT 가 챔버에서 광선을 어떻게 튕기는가 — 그리고 무엇을 믿을 수 있는가?**

### ⚡ 결론부터 (TL;DR)

1. **RT 는 환경에서 정확하다.** TX→바닥→RX 반사를 Sionna 가 독립적으로 **19.31 ns / -14.68 dB** 로 찾았고, 거울상+프레넬 손계산(19.31 ns / -14.68 dB)과 **0.00 dB / 0.00 ns** 로 일치한다. → **이게 우리가 RT 를 믿는 근거다.**
2. **그런데 RT 는 표적을 못 본다.** 드론이 씬에 **있는데도** 정반사 solver 가 찾은 표적 경유 경로는 depth 3 까지 **0개**다.
3. **광선을 400M 발 쏴도 수렴하지 않는다.** 광선을 16배로 늘리면 코히어런트 합이 **+14.2 dB** 계속 커진다. 비코히어런트 합은 **-73.1 dB** 로 수렴하지만, 그건 σ 가 아니라 **산란계수 S 의 함수**다.
4. **결정적 실험 [D]**: 금속 평판의 변을 0.2 → 4 m 로 키우면 σ 는 **+52 dB** 변하는데, RT 진폭비는 **0.00 dB** 움직인다 (시드 산포 0.00 dB). 값은 image-source 예측 -7.88 dB 그대로다. **표적 크기가 어디에도 안 들어간다.**
5. **왜인가**: ITU `metal` 의 산란계수는 **S = 0.0** 이고, SBR 로 재면 σ 의 **59%** 가 그 금속 부품(모터·배터리·PCB·카메라)에서 나온다. **σ 는 적분에서 나온다** — 산란적분 단계가 없는 전파용 path solver 에서는 창발하지 않는다.
6. **그래서 하이브리드**: 환경 = Sionna RT, 표적 = SBR. 그리고 표적 경유 **바닥 유령**이 진짜 표적 대비 **+3.51 m / -18.2 dB** 에 서고, 5G 대역(ΔRb 3.05 m)에서 **1.15배**라 **별개 표적으로 분해된다.**

### 🗺️ 어디부터 읽나

| 절 | 무엇을 |  |
|---|---|---|
| §1 | 광선추적이란 무엇인가 — max_depth 0/1/2/3 을 눈으로 | **렌더가 주인공.** 1-bounce 에 바닥이 나타난다 |
| §2 | 바닥 반사 — 손계산 vs RT | **RT 를 믿는 근거**. 0.00 dB 일치 |
| §3 | RT 는 표적을 못 본다 — 4억 발 반증 [A]~[E] | **[D] 가 결정적**. 몬테카를로 잡음 없는 깨끗한 반증 |
| §4 | 하이브리드 + 바닥 유령 | 다음 단계의 숙제 |
| §5 | 무엇을 믿고 무엇을 믿지 않나 | 신뢰 경계 |

---


## 🔰 5분이면 이해하는 이 리포트

패시브 레이더로 드론을 잡으려면 먼저 알아야 할 게 있습니다 — **드론이 전파를 얼마나 밝게 되비추는가**(이걸 RCS, 레이더 되비침 밝기라고 부릅니다). 우리가 쓰는 시뮬레이터 Sionna 는 전파가 방 안에서 어떻게 튕기는지를 **광선추적**(빛줄기를 사방으로 쏴서 어디에 부딪히나 하나하나 따라가는 방식)으로 계산합니다. 그럼 그 광선추적만으로 드론의 되비침 밝기까지 알 수 있을까요? 이 리포트는 바로 그 질문에 답합니다.

이해하는 방법은 이렇습니다. 광선추적은 **깜깜한 방에서 손전등 빛줄기를 사방으로 쏘며 어디에 부딪히나 지켜보는 것**과 같습니다. 벽·바닥처럼 크고 가까운 면은 빛줄기가 잘 맞습니다. 그런데 방 저편에 떠 있는 작은 드론은 손전등 빛이 거의 안 맞습니다 — 너무 작고 멀거든요.

그래서 빛줄기를 **4억 발**까지 쏴 봤습니다(GPU 가 놀고 있었으니까요). 결과는 둘로 갈립니다. **방(房)은 정확하게 봅니다** — 바닥에 튕겨 돌아오는 반사를 손으로 푼 계산과 소수점까지 똑같이 맞혔습니다. 그래서 방에 대해서는 Sionna 를 믿습니다. **하지만 드론의 되비침 밝기(RCS)는 못 냅니다.** 빛줄기를 아무리 늘려도 값이 한 곳에 멈추지 않고 계속 커집니다. 이유는 간단합니다 — 되비침 밝기는 표적 표면을 **쭉 훑어 더하는(적분) 계산**에서 나오는데, Sionna 의 전파용 광선추적에는 그 '더하기' 단계가 아예 없습니다. 밝기를 재려면 그 적분을 해 주는 다른 도구(SBR)가 필요합니다.

한 가지 더 있습니다. 바닥 반사는 표적을 따라다니는 **유령**을 만듭니다(TX→드론→바닥→RX 로 한 번 더 도는 가짜 메아리). 이 유령은 진짜 드론처럼 움직여서, 5G 처럼 거리를 잘 가르는 신호는 오히려 이 유령을 **별개의 가짜 표적**으로 착각합니다.

**그래서 결론은 이렇습니다.** 방은 Sionna RT 에게 맡기고, 표적의 되비침 밝기는 SBR 에게 맡기는 **하이브리드**로 갑니다. 그리고 바닥 유령은 다음 단계에서 반드시 다뤄야 할 숙제로 남습니다. 아래 본문은 이 이야기를 수식과 측정 수치로 하나하나 증명합니다 — 쉬운 요약은 여기까지고, 지금부터가 근거입니다.

## 📋 이 결과가 어디서 어떻게 나왔나

> 이 절은 **직접 참여하지 않은 사람도 출처를 따라가고 재현할 수 있도록** 넣었습니다. 버전·GPU 는 노트북 생성 시점에 **실제로 읽어온 값**입니다.

### 1️⃣ 무엇을 참고했나

| 항목 | 출처 | 성격 |
|---|---|---|
| Sionna RT 2.0 문서 (PathSolver / RadioMapSolver / render_to_file) | https://nvlabs.github.io/sionna/rt/ | 1차 (API·의미) |
| "RCS is not supported out of the box" | Sionna 메인테이너, GitHub Discussions (NVlabs/sionna) | 1차 (도구의 범위를 만든 사람이 직접 말함) |
| ITU-R P.2040 재질(콘크리트·금속의 εr, σ) | Sionna 내장 `ITURadioMaterial` — **우리가 값을 적지 않고 Sionna 에게 물어본다** | 1차 (표준) |
| 프레넬 반사계수(TM/TE), 거울상법(image source) | 표준 전자기 교과서 (Balanis, *Advanced Engineering Electromagnetics*) | 교과서 — `benchmark/geometry.py:_fresnel_floor` 에 구현 |
| SBR (Shooting-and-Bouncing Rays) = GO 광선 + PO 표면적분 | 상용 EM 솔버(FEKO/CST/HFSS SBR+)의 표준 방법. `src/rcs_sbr.py` | 방법론 — 평판 −0.01 dB · 금속구 +0.39 dB 로 해석해 대조 검증 |
| 드론 5종 제원 (Mavic 4 Pro 외형·치수) | `docs/drone_specs_2026.json` (출처 URL 포함, 적대적 검증됨) | 1차 — **지어낸 숫자 아님** |
| 옛 리포트에서 살린 발견 | `docs/ARCHIVE.md` | 내부 |

### 2️⃣ 어떤 도구가 무엇을 했나 — **Sionna 내부인가, 우리가 짠 건가**

| 도구 | 하는 일 | 어디서 도는가 |
|---|---|---|
| `sionna-rt` | Sionna RT `PathSolver` — 전파 광선추적. 경로별 **지연 τ · 도플러 f_d · 복소이득 · 반사점 좌표**를 준다 | 🟢 **Sionna 내부** (Mitsuba 3 / OptiX, GPU) |
| `sionna-render` | Sionna RT `Scene.render_to_file()` — 씬·**추적된 광선**·라디오맵을 사진처럼 렌더 | 🟢 **Sionna 내부** (Mitsuba 3 경로추적 렌더러, GPU) |
| `sionna-radiomap` | Sionna RT `RadioMapSolver` — 공간별 전파 세기 분포 | 🟢 **Sionna 내부** (GPU) |
| `sbr` | SBR (`src/rcs_sbr.py`) — **Mitsuba 광선 + PO 표면적분**으로 RCS. 가림(occlusion) 포함 | 🟡 **우리가 짰다** — 다만 광선추적은 Sionna 가 쓰는 **Mitsuba 3 엔진 그대로** (GPU). Sionna 에 RCS 솔버가 없기 때문 |
| `trimesh-cad` | CAD 모델링 (`src/cadkit.py` + `src/drone_cad.py`) — 로프트·스윕·회전체·**불리언(CSG)** | 🔴 **별도** (trimesh + manifold3d + shapely + scipy, CPU) |
| `matplotlib` | matplotlib — 도표·그래프 | 🔴 **별도** (CPU). 계산 결과를 *그리기만* 한다 |

> 🔑 **이 구분이 이 프로젝트에서 가장 자주 오해받는 지점입니다.**
> - **전파**(경로·지연·도플러·렌더·라디오맵)는 🟢 **Sionna 가** 합니다.
> - **표적 RCS** 는 🟡 우리가 짠 **SBR** 이 합니다 — Sionna 에 RCS 솔버가 없기 때문입니다. 다만 광선추적은 Sionna 가 쓰는 **Mitsuba 3 엔진을 그대로** 씁니다.
> - **레이더 신호처리**(ECA/CFAR)는 🔴 우리가 짰습니다 — Sionna 에 레이더 DSP 가 없습니다.

### 3️⃣ 라이브러리 (실행 시점 **실측** 버전)

| 라이브러리 | 버전 | 무엇에 쓰나 |
|---|---|---|
| `sionna` | 2.0.1 | 광선추적(RT) + PHY(OFDM/NR/채널) — **이 프로젝트의 중심** |
| `mitsuba` | 3.8.0 | Sionna RT 의 렌더러·광선추적 백엔드 (OptiX, GPU). SBR 도 이걸 쓴다 |
| `drjit` | 1.3.1 | Mitsuba 의 JIT 컴파일러 — GPU 커널 생성 |
| `torch` | 2.12.1 | Sionna PHY 백엔드 — ⚠ Sionna 2.0 은 TensorFlow 가 아니라 **PyTorch** |
| `trimesh` | 4.12.2 | 메쉬 CAD·**검증** — 로프트/스윕/불리언 + watertight·법선·퇴화면 검사 |
| `numpy` | 2.5.0 | 수치 계산 전반 |
| `scipy` | 1.18.0 | 스플라인(단면 보간·암 경로) · STFT(스펙트로그램) |
| `matplotlib` | 3.11.0 | 도표 |

### 4️⃣ 어디서 돌렸나

- **Python** 3.12.13 · Linux 5.15.0-136-generic
- **GPU** — `src/gpu.py` 가 **여유 메모리를 보고 자동 선택**합니다 (하드코딩 없음):
  - 0, NVIDIA GeForce RTX 4090, 24564 MiB, 580.126.09
  - 1, NVIDIA GeForce RTX 4090, 24564 MiB, 580.126.09
  - 2, NVIDIA GeForce RTX 4090, 24564 MiB, 580.126.09
  - 3, NVIDIA GeForce RTX 4090, 24564 MiB, 580.126.09
- `CUDA_VISIBLE_DEVICES` = (고정 안 함 — src/gpu.py 가 여유 메모리 보고 자동 선택)

- **계산 비용**: GPU 1장(`src/gpu.py` 가 여유 메모리 보고 자동 선택). 측정 ~20초 (광선 최대 400M × 시드 3개), Sionna 렌더 19장 ~80초 (num_samples=640, 1760x1200). 전체 ~3분.

### 5️⃣ 어떻게 다시 돌리나 (재현)

```bash
# 0) 환경
cd /home/yunjung/workspace/sionna2

# 1) 측정만 (모든 숫자 → outputs/report3_rt.json)
~/.venvs/py312/bin/python benchmark/rt_experiments.py
~/.venvs/py312/bin/python benchmark/rt_experiments.py --quick    # 짧게
~/.venvs/py312/bin/python benchmark/rt_experiments.py --only depth,floor   # 일부만

# 2) 전체 (측정 → Sionna 렌더 → 그림 → 이 노트북)
~/.venvs/py312/bin/python src/build_report3.py
~/.venvs/py312/bin/python src/build_report3.py --no-render   # 렌더 재사용
```

### 6️⃣ 본문 숫자는 어디서 오나

이 노트북의 **숫자는 손으로 적지 않았습니다.** 측정 스크립트가 JSON 을 남기고, 노트북 생성기(`src/make_notebook*.py`)가 그 JSON 을 읽어 본문에 주입합니다. → **그림과 글이 어긋날 수 없습니다.** 숫자가 이상하면 JSON 을 보세요.

### 7️⃣ 무엇이 산출되나

| 산출물 | 무엇 |
|---|---|
| `outputs/report3_rt.json` | **이 리포트의 모든 숫자.** §1~§4 측정 전부 |
| `outputs/renders/report3/*.png` | **Sionna 가 렌더한 19장** — 씬 6 · 경로 9 · 라디오맵 4 |
| `outputs/figures/report3_f1..f8.png` | 본문 그림 8장 (렌더 조립 + 실험 그래프) |
| `benchmark/rt_experiments.py` | 측정 스크립트 (재현 가능한 실험 [A]~[E]) |

### 8️⃣ ⚠️ 믿으면 안 되는 것 (신뢰 경계)

> 정직함이 이 프로젝트의 규칙입니다. **아래는 이 리포트가 보장하지 않는 것들입니다.**

- **흡수체는 실측이 아니라 모델값이다.** `materials.py` 의 absorber 는 평평한 단일면 |Γ|=0.549 (−5.2 dB)이고, 진짜 −25~−30 dB 는 **피라미드 골짜기 다중반사로 얻는 기하 효과**이자 **설계 목표**다. 그래서 이 씬에서 RT 가 낸 흡수체 반사(-9.8 dB 천장 등)는 **실제 챔버의 값이 아니다.** 바닥(ITU 콘크리트)만이 표준 재질이다.
- **§2 의 0.00 dB 일치는 '바닥 모델이 옳다'가 아니라 '두 방법이 같은 물리를 같게 푼다'는 뜻이다.** 둘 다 같은 (εr, σ) 를 쓴다. 실측 대조가 아니다.
- **§3 은 Sionna 가 '틀렸다'는 뜻이 아니다.** Sionna 는 **전파용 도구**다. ⛔ *"레이트레이싱은 RCS 를 못 낸다"* 는 **거짓** — SBR 이 바로 레이트레이싱이고 σ 를 계산한다. ✅ 참인 명제는 좁다: **"산란적분 단계가 없는 전파용 path solver 에서는 σ 가 창발하지 않는다."**
- **SBR 의 절대 σ 불확실도는 모른다.** 평판/금속구 해석해와는 맞지만(−0.01 / +0.39 dB), 드론의 절대 σ 는 **실측 앵커링 전까지 인용 금지**. §3 의 '정답선'은 그래서 **절대값이 아니라 기울기·불변성의 기준**으로만 읽어야 한다.
- **§4 의 유령은 RT 가 아니라 손계산이다.** Sionna 정반사 solver 는 표적 경로를 0개 만들므로 유령도 못 만든다. 거울상+프레넬로 유도했다.
- **거리분해능 체제를 섞지 말 것.** §4 의 ΔRb = c/B 는 **부하 걸린 셀 + full-waveform 기준** 체제다. idle 셀에서 **상시 신호만**(5G SSB 7.2 MHz) 쓰면 ΔRb = 41.6 m 라 유령은 오히려 **표적에 묻힌다**.
- **RPM·마이크로도플러는 이 리포트에 없다** (report1/2 소관). 옛 5500 rpm 숫자를 여기서 찾지 말 것.

### 9️⃣ 앞뒤 리포트

| 리포트 | 관계 |
|---|---|
| **report1** (환경·메쉬·분절) | 이 리포트가 쓰는 **챔버·드론 메쉬**를 만든다 |
| **report2** (OFDM 파형 · RCS) | §3 이 'RT 로는 못 한다'고 증명한 **σ 를 SBR 로 실제로 계산**한다. §4 의 파형·대역폭도 거기서 온다 |
| **다음 단계** (검출) | §4 의 **바닥 유령**을 ECA/CFAR 사슬에 넣는 것이 숙제 |

<details><summary><b>🔤 용어집 — 모르는 말이 나오면 여기</b> (클릭)</summary>

| 용어 | 뜻 |
|---|---|
| **PathSolver** | Sionna RT 의 경로 탐색기. 경로별 지연 τ · 도플러 f_d · 복소이득 · 반사점을 준다 |
| **max_depth** | 광선이 **최대 몇 번 튕기는가**. 0 = 직접파만, 1 = 1회 반사까지 |
| **paths.objects** | 각 경로가 맞은 물체의 **object_id** (1부터). ⚠ `scene.objects` 의 열거 순서가 **아니다** |
| **정반사 / 확산** | specular = 거울 반사(GO). diffuse = 산란계수 S 로 흩뿌리는 확산 채널 |
| **산란계수 S** | 재질이 입사 에너지를 확산으로 얼마나 흩뿌리는가. **ITU 재질은 전부 S=0** |
| **거울상법(image source)** | 반사면 너머에 송/수신기의 '거울상'을 놓아 반사경로를 직선으로 펴는 기법 |
| **프레넬 |Γ|** | 매질 경계의 반사계수. V편파(수직)는 입사면에 평행 → **TM** 성분 |
| **RCS (σ)** | 레이더 반사 단면적 [m²]. dBsm = 10·log₁₀(σ/1 m²) |
| **SBR** | Shooting-and-Bouncing Rays = **광선(GO)으로 조명면을 찾고 그 위에서 PO 적분**. σ 를 준다 |
| **semi-anechoic** | **벽·천장만** 흡수체이고 **바닥은 반사성**인 챔버. 우리 챔버가 이것 |
| **ECA** | 직접파(TX→RX 가시선)를 지우는 전처리. **도플러 0** 인 것만 지운다 → 유령은 못 지운다 |
| **ΔRb** | 바이스태틱 거리분해능 = c/B. 두 표적이 이보다 멀면 **따로** 보인다 |

</details>

---


## §0. 무대 — 무엇을 어디에 놓았나

> 🔍 **여기서 하는 일:** 실험 무대를 세팅합니다 — 방·신호원(TX)·수신기(RX)·드론을 각각 어디에 뒀는지 좌표로 못 박습니다.

**semi-anechoic 챔버 30 x 20 x 11 m.** 벽 4면 + 천장은 피라미드 흡수체, **바닥은 반사성 콘크리트**(ITU `concrete`). 방 안에서 **유일하게 강한 반사면이 바닥**이다.

| | 위치 [m] | 비고 |
|---|---|---|
| **TX** (신호원) | (4.0, 2.5, 8.0) | 한쪽 벽 상단 |
| **RX** (패시브 수신) | (4.0, 17.5, 6.5) | 같은 벽, y 로 15.1 m 떨어짐 |
| **표적** (Mavic 4 Pro) | (21.0, 10.0, 5.5) | quiet zone, 0.35 m 급 |

- 직접파 **L = 15.07 m** (τ = 0.05 ns)
- TX→표적 **R1 = 18.75 m**, 표적→RX **R2 = 18.61 m**, 바이스태틱 각 **β = 47.6°**
- 반송파 **3.5 GHz** · Sionna **2.0.1**

> 💡 **TX · RX · 바닥 반사점이 모두 x = 4 m 평면 위에 있다.** 그래서 아래 렌더의 카메라는 그 평면을 정면으로 본다 — 바닥 반사 삼각형이 찌그러지지 않는다.

![chamber gallery](outputs/figures/report3_f3_gallery.png)

---
## §1. 광선추적이란 무엇인가 — 눈으로

> 🔍 **여기서 하는 일:** 광선추적이 뭔지 눈으로 봅니다 — 빛줄기를 몇 번까지 튕기게 허락하느냐에 따라 무엇이 새로 보이는지(직접파 → 바닥 → 이중반사) 한 장씩 넘겨 봅니다.

`PathSolver`(Sionna 의 경로 탐색기) 에 **`max_depth`(광선이 최대 몇 번 튕기게 할지) 하나만** 바꿔가며 돌린다. 광선이 몇 번까지 튕겨도 되는가?

| `max_depth` | 경로 수 | 무엇이 늘었나 |
|---|---|---|
| 0 | **1** | 직접파(LOS)뿐 |
| 1 | **4** | + 천장 · 왼쪽 벽 · **바닥** |
| 2 | **16** | + 2회 반사 (흡수체 피라미드 **골짜기 안의 이중반사**) |
| 3 | **40** | + 3회 반사 |

**1-bounce 에서 바닥이 나타난다.** 이게 이 리포트의 첫 그림이다.

![max_depth 0/1/2/3](outputs/figures/report3_f1_bounces.png)

### 경로 원장 — 무엇을, 언제, 얼마나, **무엇을 맞고**

`paths.objects` 로 각 경로가 **어느 물체를 맞았는지** 이름을 얻는다.

**max_depth = 1 의 네 경로 (전부 실측):**

| 지연 (LOS 대비) | 진폭 (LOS 대비) | 도플러 | 맞은 물체 | 반사점 [m] |
|---|---|---|---|---|
| 0.00 ns | +0.00 dB | +0.0 Hz | `LOS (직접파)` | - |
| 12.97 ns | -9.82 dB | +0.0 Hz | `absorber_ceiling` | [8.588, 9.415, 10.749] |
| 14.90 ns | -17.16 dB | +0.0 Hz | `absorber_left` | [0.242, 10.716, 2.259] |
| 19.31 ns | -14.68 dB | +0.0 Hz | `floor_light` | [4.0, 10.776, 0.0] |

> ⚠️ **여기서 한 번 데일 뻔했다.** `paths.objects` 가 주는 건 **object_id** 이고 **1부터 시작**한다. `scene.objects` dict 의 **열거 순서(0부터)** 로 매핑하면 **전부 한 칸씩 밀려서** 천장 반사를 `backing_front`(앞벽) 라고 부르게 된다. 반드시 `{int(o.object_id): name}` 으로 역맵을 만들 것 (`benchmark/rt_experiments.py: id_to_name`).

> 그리고 — **드론이 씬에 있는데도** depth 3 까지 표적을 경유한 경로는 **0개**다. 이 사실이 §3 이다.

![path ledger](outputs/figures/report3_f2_ledger.png)

---
## §2. 바닥 반사 — RT 가 옳았다

> 🔍 **여기서 하는 일:** Sionna 를 전혀 안 쓰고 손으로 푼 바닥 반사 값과, Sionna 가 스스로 찾아낸 값을 나란히 놓고 맞대 봅니다 — 둘이 얼마나 똑같은지가 곧 'RT 를 믿어도 되는 근거'입니다.

**이 절이 우리가 RT 를 믿는 근거다.** Sionna 를 전혀 안 쓰고 손으로 계산한 값과 대조한다.

### 손계산 (거울상 + 프레넬)
1. 바닥(z=0)에 **RX 의 거울상** RX' = (…, −6.5) 을 놓는다 → 반사경로가 직선으로 펴진다.
2. 직접파 **15.07 m**, 바닥 경유 **20.86 m** → 여분지연 **19.31 ns**
3. 입사각 **46.0°** (법선 기준). → **스침각이 아니다** (바닥면 위로 44.0°).
4. V편파 = 입사면 평행 = **TM** → ITU 콘크리트(εr=5.24, σ=0.1231 S/m) 에서 **|Γ| = 0.255**  (참고: TE 면 0.518)
5. 확산손 20·log10(L/Lf) = **-2.8 dB**

→ 직접파 대비 **-14.68 dB**

### Sionna RT 가 독립적으로 찾은 경로
→ **19.31 ns / -14.68 dB** (`floor_light`, 반사점 [4.0, 10.776, 0.0])

## → **0.00 dB · 0.00 ns 일치**

**RT 는 환경에서 정확하다.** 지연도, 입사각도, 프레넬 계수도, 확산도 우리가 손으로 푼 것과 같다.

![floor bounce verification](outputs/figures/report3_f4_floor.png)

### 같은 지연에 있던 **두 번째 경로**의 정체 — 규명했다

RT 는 ≈19.3 ns 에 경로를 **두 개** 낸다. 하나는 위의 바닥 반사(-14.68 dB), 다른 하나는 **더 센** -11.63 dB 짜리다. `benchmark/verify_floor_ghost.py` 는 이걸 *'미규명'* 으로 남겨 뒀었다.

`paths.objects` + 반사점 좌표로 보면 답이 나온다:

| | 값 |
|---|---|
| 지연 | 19.33 ns |
| 진폭 | -11.63 dB |
| 맞은 물체 | `absorber_front` → `absorber_front` (**같은 물체를 두 번**) |
| 반사점 1 | [6.011, 0.015, 7.77] |
| 반사점 2 | [5.979, 0.028, 7.767] |

두 반사점이 **y ≈ 0.0 m 에서 3 cm 떨어져** 있다 → **앞벽 흡수체 피라미드의 골짜기 안에서 일어난 이중반사**다. 인접한 두 피라미드 면이 만드는 **다이헤드럴(코너)** 에 광선이 갇혔다가 되돌아온 것이다.

> 🔍 **이건 흡수체가 작동하는 바로 그 원리**(골짜기 다중반사로 에너지를 갉아먹는다)를 RT 가 현장에서 잡은 것이다. 다만 **우리 흡수체는 모델값**이라 한 번 튕길 때마다 −5.2 dB 밖에 안 깎인다 → 두 번 튕기고도 살아나온다. **진짜 흡수체(−25 dB급)라면 이 경로는 없다.** (→ 신뢰 경계 참조)

### 라디오맵 — 에너지가 실제로 어디로 가나

`RadioMapSolver` 로 바닥면(z=0.05 m)과 드론 평면(z=5.5 m)의 전계 분포를 본다. 드론이 **그림자를 드리운다**.

![radio maps](outputs/figures/report3_f5_radiomap.png)

---
## §3. 그런데 RT 는 **표적을 못 본다** — 광선을 4억 발 쏴도

> 🔍 **여기서 하는 일:** '광선을 더 쏘면 되비침 밝기(RCS)가 나오지 않나?'라는 반론에 4억 발로 정면 대응합니다 — 다섯 가지 실험 [A]~[E] 로 왜 안 나오는지 하나씩 못 박습니다.

정당한 반론부터: *"광선을 적게 쏴서 그런 것 아닌가? GPU 가 놀고 있으니 4억 발 쏴 보자."*

**쏴 봤다.** 다섯 가지를 잰다 (전부 `benchmark/rt_experiments.py` 로 재현 가능).

![five experiments](outputs/figures/report3_f6_no_sigma.png)

### [A] 광선 예산 25M → 400M (16배) — 수렴하는가?

| 광선 | 표적 경로 수 | 코히어런트 합 | 비코히어런트 합 |
|---|---|---|---|
| 25M | 5.3 | **-68.62 dB** (±2.67) | -74.67 dB (±1.62) |
| 50M | 12.7 | **-63.25 dB** (±0.50) | -73.47 dB (±0.43) |
| 100M | 21.7 | **-61.58 dB** (±0.39) | -73.75 dB (±0.16) |
| 200M | 48.0 | **-57.67 dB** (±0.98) | -73.34 dB (±0.80) |
| 400M | 94.3 | **-54.42 dB** (±0.20) | -73.11 dB (±0.34) |

- **코히어런트 합**(위상까지 더한 것 — 레이더 수신기가 실제로 하는 일)은 **+14.2 dB 계속 커진다.** 4배마다 약 +7 dB — 경로 수가 광선 수에 비례해 늘고 위상이 정렬돼 있어 **√N 으로 발산**한다. 수렴할 곳이 없다.
- **비코히어런트 합**은 **-73.1 dB** 로 수렴**한다**. 정직하게 말하자. 하지만 그 수렴값은 σ 가 아니다 — **[B] 가 보여주듯 산란계수 S(재질이 입사 에너지를 사방으로 얼마나 흩뿌리는지 나타내는 계수)의 함수**이고, 레이더방정식이 요구하는 값(-58.98 dB)과 **-14.1 dB** 어긋나 있다.

**쉽게 말하면 —** 광선을 두 배로 쏘면 표적에 맞는 광선도 두 배가 되니 위상까지 더한 합이 멈추지 않고 계속 불어납니다. 즉 **'진짜 값'이라 부를 멈춤점이 없습니다.** 위상을 무시하고 더한 합은 멈추긴 하지만, 그 값은 표적의 밝기가 아니라 **우리가 재질표에 적어 넣은 흩뿌림 정도(S)가 정한 값**일 뿐입니다.

> ⚠️ 시드를 3개 돌렸다. 확산은 몬테카를로다 — 시드 산포를 안 재면 잡음을 추세로 오독한다.

### [B] 산란계수 S 를 0.5x → 2.0x

| S 배수 | plastic 의 S | 코히어런트 합 |
|---|---|---|
| 0.5x | 0.10 | **-74.75 dB** (±1.07) |
| 1.0x | 0.20 | **-61.58 dB** (±0.39) |
| 2.0x | 0.40 | **-48.58 dB** (±0.55) |

→ S 를 4배로 돌리면 진폭이 **+26.2 dB** 움직인다.

**S 는 드론의 물성이 아니다. 우리가 재질표에 적어 넣는 노브다.** RT 확산으로 σ 를 '맞추는' 것은 예측이 아니라 **피팅**이다.

### [C] ITU `metal` 의 S = 0 — σ 의 대부분이 확산 채널에 **기여 0**

Sionna 에게 직접 물어봤다: ITU `metal` 의 `scattering_coefficient` = **0.0**. 드론 7개 부위 중 **4개**가 S = 0 이다.

| 부위 | 재질 | Sionna 가 든 S |
|---|---|---|
| `body` | `plastic` | **0.20** |
| `canopy` | `plastic` | **0.20** |
| `prop` | `prop_plastic` | **0.20** |
| `motor` | `metal` | **0.00** |
| `camera` | `camera_assembly` | **0.00** |
| `battery` | `metal` | **0.00** |
| `pcb` | `pcb` | **0.00** |

그런데 **같은 메쉬·같은 재질**을 SBR(광선 + PO 적분)로 재면:

| | σ (방위평균, dBsm) |
|---|---|
| 전체 | **-20.70** |
| 금속 부품만 (S = 0) | **-22.97** → 전체의 **59%** |
| 비금속만 (S > 0) | -25.59 |

> **σ 의 59% 가, RT 의 확산 채널에 기여가 정확히 0 인 부품(모터·배터리·PCB·카메라 하우징)에서 나온다.**
> ⇒ RT 확산이 보는 표적과 물리가 보는 표적은 **다른 표적**이다.

**쉽게 말하면 —** 드론에서 제일 잘 되비추는 부품(모터·배터리·금속판)을 RT 의 흩뿌림 계산은 '0'으로 봅니다. 정작 밝기의 대부분을 만드는 부분을 RT 는 아예 안 보고 있는 셈입니다.

### [D] **결정적 실험** — 금속 평판의 변을 0.2 → 4 m

여기엔 몬테카를로 잡음이 **없다**: 확산을 끄고 **정반사(거울처럼 입사각 그대로 튕기는 반사)만** 켠다. 평판은 TX·RX 이등분선에 법선을 맞춰 정반사 조건을 정확히 만족시킨다.

| 변 [m] | 면적 [m²] | 평판의 진짜 σ [dBsm] | RT 진폭비 | 시드 산포 |
|---|---|---|---|---|
| 0.2 | 0.04 | +4.4 | **-7.91 dB** | 0.00 dB |
| 0.4 | 0.16 | +16.4 | **-7.91 dB** | 0.00 dB |
| 0.8 | 0.64 | +28.5 | **-7.91 dB** | 0.00 dB |
| 1.5 | 2.25 | +39.4 | **-7.91 dB** | 0.00 dB |
| 2.5 | 6.25 | +48.3 | **-7.91 dB** | 0.00 dB |
| 4.0 | 16.00 | +56.4 | **-7.91 dB** | 0.00 dB |

## σ 는 **+52 dB** 변했다. RT 는 **0.00 dB** 움직였다.

그리고 그 불변값 **-7.91 dB** 는 정확히 image-source 예측

$$ 20\log_{10}\frac{L}{R_1+R_2} = -7.88\ \text{dB} $$

**표적의 크기가 이 식 어디에도 들어가지 않는다.** GO(기하광학)는 표면을 **국소 무한 거울**로 본다 — 거울의 '넓이'라는 개념이 없다. 그래서 면적 항이 없다.

**쉽게 말하면 —** 평판을 20배 키워 진짜 되비침 밝기를 **+52 dB**(수만 배) 바꿨는데 RT 값은 사실상 꿈쩍도 안 했습니다. RT 는 거울을 '무한히 큰 거울'로 취급해서, 그 거울이 손바닥만 한지 문짝만 한지를 **애초에 구분하지 못합니다.** 크기 정보가 계산에 들어갈 자리가 없는 겁니다.

> 🎯 **이게 이 리포트에서 가장 깨끗한 반증이다.** 잡음 0.00 dB, 시드 산포 0.00 dB. 해석의 여지가 없다.

### [E] 물리적으로 옳은 PEC 금속구 (S = 0) — 디스코볼 문제

(PEC = 완전도체 — 전파를 100% 되튕기는 이상적 금속.) 금속구의 σ 는 해석적으로 안다: σ = πr². 반경 0.30 m 면 **-5.5 dBsm** — 드론보다 **+15 dB** 크다. RT 는 이걸 볼 수 있을까?

| 광선 | 표적 경로 수 |
|---|---|
| 1M | **0** |
| 10M | **0** |
| 100M | **0** |
| 400M | **0** |

**경로가 0개다.** 광선을 400M 발 쏴도 0개다.

- **대조군**: 같은 자리·같은 재질의 **평판**(0.6 m)은 광선 1M 발로도 즉시 **1개** (-7.91 dB) 잡힌다. ⇒ **메쉬·재질·솔버는 정상이다.**
- 물리적으로 옳게(PEC = S 0) 놓으면 확산 채널이 닫히고, 남은 건 정반사뿐인데 **image-source 법은 곡면 위의 정반사점을 못 찾는다**. 구를 아무리 잘게 쪼개도 각 면이 RX 를 정확히 겨눌 확률은 0 이다 (디스코볼 문제).

**쉽게 말하면 —** 반짝이는 미러볼(디스코볼)처럼, 둥근 금속구는 표면의 딱 한 점만 수신기 쪽으로 빛을 되쏩니다. 광선을 무작위로 뿌리면 하필 그 한 점을 정확히 때릴 확률이 사실상 0 이라, 정반사만 켠 RT 눈에는 그 큰 구가 **통째로 안 보입니다.**

### §3 결론 — **σ 는 적분에서 나온다**

다섯 실험이 같은 곳을 가리킨다:

| 실험 | 결과 | 뜻 |
|---|---|---|
| [A] 광선 16배 | 코히어런트 **+14.2 dB** 발산 | 수렴 목표가 없다 |
| [B] S 4배 | **+26.2 dB** | 드론이 아니라 **노브**의 함수 |
| [C] ITU metal | **S = 0.0** | σ 의 **59%** 가 확산 기여 0 |
| [D] 평판 σ **+52 dB** | RT **0.00 dB** | 크기가 식에 없다 |
| [E] PEC 구 | 경로 **0개** | 정반사만으론 곡면을 못 본다 |

> **Sionna 가 틀린 게 아니다. Sionna 는 전파용 도구다.** 메인테이너도 그렇게 말한다 — *"RCS is not supported out of the box"*.

#### ⛔ 쓰면 안 되는 문장 / ✅ 참인 문장

| | |
|---|---|
| ⛔ **거짓** | *"레이트레이싱은 RCS 를 못 낸다"* — **SBR 이 바로 레이트레이싱이고 σ 를 계산한다.** 상용 EM 솔버(FEKO/CST/HFSS SBR+)가 하는 일이 그것이다 |
| ✅ **참** | *"**산란적분 단계가 없는** 전파용 path solver 에서는 σ 가 창발하지 않는다."* |

GO 는 표면을 국소 무한 거울로 본다 → **면적 항이 없다**. PO 는 표면 전류를 **적분**한다 → 면적이 들어온다. **차이는 적분 한 단계다.**

---
## §4. 그래서 하이브리드 — 환경 = RT, 표적 = SBR

> 🔍 **여기서 하는 일:** §2·§3 의 결론을 합쳐 역할을 나눕니다(방 = RT, 표적 = SBR). 그리고 바닥이 만드는 '유령'이 어디에 서고 왜 안 지워지는지, 어떤 신호에서 문제가 되는지 봅니다.

§2 와 §3 을 합치면 설계가 강제된다. 어느 엔진도 다른 쪽의 대체재가 아니다.

| | 엔진 | 근거 |
|---|---|---|
| **환경** (직접파·바닥·벽·천장, 지연·도플러·기하) | 🟢 **Sionna RT** | §2: 손계산과 **0.00 dB** 일치 |
| **표적** (σ) | 🟡 **SBR** (Mitsuba 광선 + PO 적분) | §3: RT 는 못 준다. SBR 은 평판 −0.01 dB · 금속구 +0.39 dB 로 검증됨 |

![hybrid split](outputs/figures/report3_f7_hybrid.png)

### 표적 경유 **바닥 유령** — 도플러가 실려 ECA 를 통과한다

**TX → 표적 → 바닥 → RX.** 이 경로는 **표적을 거치므로 표적과 함께 도플러가 실린다.** 정적 클러터와 달리 **ECA(직접파·0-도플러 제거)의 영공간 밖**이라 **안 지워진다.**

**쉽게 말하면 —** 바닥에 한 번 더 튕겨 돌아오는 메아리인데, 표적을 거쳐 왔기 때문에 표적처럼 **'움직입니다'.** 정지한 배경만 지우는 청소기(ECA)는 움직이는 건 못 지우니, 이 유령은 살아남아 수신기에 도달합니다.

거울상 + 프레넬로 유도하면 (RT 는 표적 경로를 못 만드니 손계산이다):

| | 진짜 표적 | 유령 |
|---|---|---|
| 바이스태틱 거리 Rb | 22.28 m | **25.79 m** (+3.51 m) |
| 도플러 f_d | +64.9 Hz | **+54.9 Hz** (-10.1 Hz) |
| 진폭 | 0 dB (기준) | **-18.2 dB** |
| 바닥 입사각 | — | 57.1° (|Γ| = 0.147) |

표적 속도 (-3.0, 2.0, 0.5) m/s 기준. **거의 순수 기하** — 파형과 무관하다.

### 운명을 가르는 건 **거리분해능**이다

| 파형 | B | ΔRb = c/B | 분리 / ΔRb | 결과 |
|---|---|---|---|---|
| 5G NR 100MHz | 98.3 MHz | 3.05 m | **1.15x** | ★ **별개 표적으로 분해** |
| WiFi 802.11ac | 75.6 MHz | 3.96 m | **0.89x** | 병합(묻힘) |
| LTE 20MHz | 18.0 MHz | 16.66 m | **0.21x** | 병합(묻힘) |

> ### ⚠️ **광대역의 장점이 그대로 오경보로 되돌아온다.**
> 5G 를 최고의 조명원으로 만든 바로 그 거리분해능(ΔRb = 3.05 m)이, 유령을 진짜와 **1.15배**로 갈라놓아 **별개의 (가짜) 표적**으로 세운다.

**단, 체제를 섞지 말 것.** 위 표는 **부하 걸린 셀 + full-waveform 기준신호** 체제다. 패시브 레이더의 기본인 **idle 셀 + 상시 기준신호만** 쓰는 체제에서는 점유대역이 훨씬 좁다:

| 파형 | 상시 기준신호 대역 | ΔRb | 분리 / ΔRb |
|---|---|---|---|
| 5G NR 100MHz | 7.2 MHz | 41.6 m | 0.08x **(묻힘)** |
| WiFi 802.11ac | 76.6 MHz | 3.9 m | 0.90x **(묻힘)** |
| LTE 20MHz | 18.0 MHz | 16.7 m | 0.21x **(묻힘)** |

→ 5G SSB(7.2 MHz)만 쓰면 ΔRb = 41.6 m 라 유령은 **표적에 묻힌다.** 유령 문제는 **광대역 기준신호를 쓸 때** 생긴다.

> 📌 **다음 단계의 숙제**: 이 유령을 ECA/CFAR 사슬에 실제로 주입해 Pd/오경보를 재는 것. 가장 확실한 대책은 **AoA** — 유령은 **바닥에서 오므로 도래 고각이 음수**다.

![floor ghost](outputs/figures/report3_f8_ghost.png)

---
## §5. 정리 — 무엇을 믿고 무엇을 믿지 않나

> 🔍 **여기서 하는 일:** 이 리포트에서 무엇을 믿어도 되고(방·경로·라디오맵) 무엇은 못 믿는지(표적 σ·흡수체 레벨) 한 표로 매듭짓습니다 — 그리고 밟았던 함정들을 남겨 둡니다.

| 대상 | 판정 | 근거 (측정) |
|---|---|---|
| **환경 전파** (지연·도플러·기하·다중경로) | ✅ **믿는다** | 바닥 반사가 손계산과 **0.00 dB / 0.00 ns** 일치 |
| **경로의 정체** (`paths.objects`) | ✅ **믿는다** | 19.3 ns 의 미규명 탭을 **앞벽 흡수체 이중반사**로 규명 |
| **라디오맵** (공간 분포·그림자) | ✅ **믿는다** (상대적으로) | 같은 solver, 같은 물리 |
| **표적 σ** | ❌ **안 믿는다** | [A]~[E] 다섯 실험. 특히 [D]: σ +52 dB 에 RT 0.00 dB |
| **흡수체 반사 레벨** | ⚠️ **모델값이다** | −25 dB 는 **설계 목표**이지 이 씬의 측정값이 아니다 |
| **바닥 유령의 검출 결과** | ⚠️ **아직 안 쟀다** | 기하만 유도했다 (+3.51 m / -18.2 dB). Pd·오경보는 다음 단계 |

### 한 문장으로

> **Sionna RT 는 방(房)을 정확히 안다. 표적은 모른다.**
> 방은 RT 에게 맡기고, 표적은 **적분(SBR)** 에게 맡긴다.

### 이 리포트가 남긴 재현 가능한 것들
- `benchmark/rt_experiments.py` — 실험 [A]~[E] + §1·§2·§4 측정. 한 번 돌리면 `outputs/report3_rt.json` 이 다시 만들어지고 이 노트북의 **모든 숫자가 갱신된다.**
- `src/build_report3.py` — Sionna 렌더 19장 + 그림 8장 + 이 노트북.

### 알아 둘 함정 (이 리포트에서 실제로 밟은 것들)
1. **`paths.objects` 는 object_id(1부터)** — dict 열거 순서로 매핑하면 라벨이 전부 밀린다.
2. **닫힌 방은 새까맣게 렌더된다** — 환경광이 못 들어온다. `clip_at` 으로 천장을 열어야 보인다 (전파 계산과는 무관한 **렌더 조명 트릭**).
3. **`cutaway` 는 씬을 바꾼다** — 앞벽을 지우면 경로가 16 → 13개로 줄고, 하필 19.3 ns 의 그 이중반사가 **앞벽 흡수체**라서 사라진다. **경로 렌더는 측정과 같은 씬(cutaway 없이)** 에서 해야 한다.
4. **Sionna 의 경로 렌더는 굵기를 못 바꾼다** (반경 min(0.20, 0.005·씬) 하드코딩). 경로 10개가 넘으면 그림이 포화된다 → 원장(표/그래프)으로 읽어야 한다.
5. **`RadioMaterial.scattering_coefficient` 는 `drjit.cuda.ad.Float`** — `float()` 에 바로 넣으면 TypeError.